In [ ]:
from pathlib import Path

import h5py
import numpy as np
import torch
from torch.utils.data import DataLoader

import fluidbench
from fluidbench.cfdataset import CFDataset

In [ ]:
ID = "cylinder_Re41_interp"
dataset_path = fluidbench.download(ID)

In [ ]:
def load_split(split):
    folder = dataset_path / split
    cases = sorted((path for path in folder.glob("*.h5") if path.stem != "constants"), key=lambda path: int(path.stem))
    data = []
    for case in cases:
        with h5py.File(case, "r") as file:
            data.append((file["inputs/scalars"][:], file["inputs/fields"][:], file["targets/fields"][:]))
    return data

def to_dataset(cases):
    scalars, fields, targets = (np.concatenate(arrays) for arrays in zip(*cases))
    return CFDataset(scalars, fields, targets, np.arange(len(targets)))

In [ ]:
### Model definition
from minimal_models import BaselineZero as architecture

weights_path = Path("weights") / ID / f"{architecture.__name__}.pt"

model = architecture()
model.load_weights(weights_path)
model.to(model.device)

In [ ]:
# Metrics are calculated by fluidbench.run() from the benchmark test split.
submission_config = {
    "datasetId": ID,
    "id": "baseline-zero",
    "model": "Baseline Zero",
    "authors": "Pablo A.C.",
    "year": 2026,
    "representation": "Grid",
}

In [ ]:
submission = fluidbench.run(model, submission_config)